In [0]:

from pyspark.sql import functions as F
from datetime import datetime, timezone
import uuid

CATALOG = "workspace"
SCHEMA = "codi_bronze"

TARGET_TABLE = (
    f"{CATALOG}.{SCHEMA}.deis_egresos_raw"
)

# Reemplazar con la ruta exacta del archivo
SOURCE_PATH = (
    "/Volumes/workspace/codi_bronze/"
    "raw_files/NOMBRE_ARCHIVO.csv"
)

# Ajustar según el separador real del CSV
DELIMITER = ";"

BATCH_ID = str(uuid.uuid4())

print("Destino:", TARGET_TABLE)
print("Lote:", BATCH_ID)


In [0]:

import requests

URL = (
    "https://repositoriodeis.minsal.cl/"
    "DatosAbiertos/EGRESOS/EGRESOS_2020.zip"
)

try:
    response = requests.get(
        URL,
        headers={"Range": "bytes=0-3"},
        timeout=20,
        stream=True
    )

    print("HTTP:", response.status_code)
    print(
        "Content-Type:",
        response.headers.get("Content-Type")
    )
    print(
        "Content-Length:",
        response.headers.get("Content-Length")
    )

    if response.status_code in (200, 206):
        print("Servidor accesible.")
    else:
        print("Respuesta HTTP no satisfactoria.")

    response.close()

except requests.RequestException as e:
    print("No fue posible acceder al servidor.")
    print(type(e).__name__, str(e)[:500])


In [0]:

import requests
import hashlib
import shutil
from pathlib import Path
from zipfile import ZipFile

BASE_URL = (
    "https://repositoriodeis.minsal.cl/"
    "DatosAbiertos/EGRESOS"
)

YEAR = 2020

BASE_PATH = Path(
    "/Volumes/workspace/codi_bronze/raw_files"
)

ZIP_DIR = BASE_PATH / "source_zip"
EXTRACT_DIR = BASE_PATH / "extracted" / str(YEAR)

ZIP_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

zip_name = f"EGRESOS_{YEAR}.zip"
zip_path = ZIP_DIR / zip_name
download_url = f"{BASE_URL}/{zip_name}"

# Descargar solo cuando aún no exista
if not zip_path.exists():

    temp_path = ZIP_DIR / f"{zip_name}.part"

    try:
        with requests.get(
            download_url,
            stream=True,
            timeout=(20, 120)
        ) as response:

            response.raise_for_status()

            with open(temp_path, "wb") as output:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        output.write(chunk)

        temp_path.replace(zip_path)

    finally:
        if temp_path.exists():
            temp_path.unlink()

    print("Descarga completada.")
else:
    print("ZIP existente, se reutilizará.")

# Validar integridad ZIP
with ZipFile(zip_path) as archive:

    invalid_file = archive.testzip()

    if invalid_file:
        raise ValueError(
            f"Archivo ZIP corrupto: {invalid_file}"
        )

    for member in archive.infolist():

        if member.is_dir():
            continue

        name = Path(member.filename).name

        if not name.lower().endswith(
            (".csv", ".xlsx", ".xls", ".txt")
        ):
            continue

        destination = EXTRACT_DIR / name

        # Extracción controlada
        # Los archivos internos se aplanan y
        # se rechazan nombres duplicados.
        if destination.exists():
            print("Ya existe:", name)
            continue

        with archive.open(member) as source:
            with open(destination, "wb") as target:
                shutil.copyfileobj(source, target)

        print("Extraído:", destination)

print("Proceso finalizado.")


In [0]:

from pathlib import Path
import csv

SOURCE_PATH = (
    "/Volumes/workspace/codi_bronze/raw_files/"
    "extracted/2020/EGRE_DATOS_ABIERTOS_2020.csv"
)

file_path = Path(SOURCE_PATH)

if not file_path.is_file():
    raise FileNotFoundError(SOURCE_PATH)

print("ARCHIVO:", file_path.name)
print("TAMAÑO MB:", round(
    file_path.stat().st_size / 1024**2, 2
))

# Leer solo una muestra, sin cargar el CSV completo
with open(file_path, "rb") as f:
    sample_bytes = f.read(65536)

encoding = None

for candidate in ("utf-8-sig", "cp1252", "latin-1"):
    try:
        sample = sample_bytes.decode(candidate)
        encoding = candidate
        break
    except UnicodeDecodeError:
        continue

if encoding is None:
    raise ValueError("Codificación no identificada")

try:
    dialect = csv.Sniffer().sniff(
        sample, delimiters=";,\t|"
    )
    delimiter = dialect.delimiter
except csv.Error:
    raise ValueError(
        "No se pudo determinar el separador. "
        "Revisar las primeras líneas."
    )

print("CODIFICACIÓN:", encoding)
print("SEPARADOR:", repr(delimiter))

with open(
    file_path, "r",
    encoding=encoding,
    newline=""
) as f:
    for i, line in enumerate(f):
        print(f"Línea {i+1}: {line[:350].rstrip()}")
        if i >= 2:
            break


In [0]:

from pyspark.sql import functions as F

spark_encoding = {
    "utf-8-sig": "UTF-8",
    "cp1252": "windows-1252",
    "latin-1": "ISO-8859-1"
}[encoding]

df_source = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("sep", delimiter)
    .option("encoding", spark_encoding)
    .option("inferSchema", "false")
    .option("mode", "FAILFAST")
    .load(SOURCE_PATH)
)

total_registros = df_source.count()
total_columnas = len(df_source.columns)

print("REGISTROS:", f"{total_registros:,}")
print("COLUMNAS:", total_columnas)

df_source.printSchema()
display(df_source.limit(10))


In [0]:

from pyspark.sql import functions as F
import uuid

TARGET_TABLE = "workspace.codi_bronze.deis_egresos_2020"

# Validación de tabla existente
if spark.catalog.tableExists(TARGET_TABLE):
    print(f"La tabla ya existe: {TARGET_TABLE}")
    print("Se omite la escritura para proteger los datos.")

else:
    batch_id = str(uuid.uuid4())

    df_bronze = (
        df_source
        .withColumn("_source_system", F.lit("DEIS"))
        .withColumn("_source_year", F.lit(2020))
        .withColumn("_source_file", F.lit(SOURCE_PATH))
        .withColumn("_batch_id", F.lit(batch_id))
        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )
    )

    # Crear tabla Delta solamente si no existe
    (
        df_bronze.write
        .format("delta")
        .saveAsTable(TARGET_TABLE)
    )

    print("Tabla Bronze creada correctamente.")
    print("Batch ID:", batch_id)

# Validar datos persistidos
bronze_count = spark.table(TARGET_TABLE).count()

print("-" * 50)
print("TABLA:", TARGET_TABLE)
print("REGISTROS BRONZE:", f"{bronze_count:,}")

if "total_registros" in globals():
    print("REGISTROS ORIGEN:", f"{total_registros:,}")
    print("DIFERENCIA:", total_registros - bronze_count)

    if total_registros != bronze_count:
        print("ADVERTENCIA: existen diferencias.")
    else:
        print("CONCILIACIÓN EXITOSA")

display(spark.table(TARGET_TABLE).limit(10))


In [0]:

DESCRIBE DETAIL workspace.codi_bronze.deis_egresos_2020;
